# Barrido de $\gamma$ en `solve_pressure` — contraste con la inestabilidad global del paper ($\gamma\approx1.12$)

## Por qué este notebook, y qué NO es

En el Bloque 2 vimos que el $\gamma_\text{crit}$ calculado punto a punto con $Z_p(x,\gamma)$ (rango 2.8–3.6 según $x/L$) **no coincide** con el $\gamma=1.12$ que reporta Neely & Kim (1986) en la Fig. 7 — y no tenía por qué coincidir: ese análisis es local (un punto aislado), mientras que la inestabilidad del paper es una propiedad del **sistema espacialmente acoplado completo** (la EDP macromecánica de fluido, Ec. 1, que acopla todas las posiciones).

En vez de construir un sistema en espacio de estados de >1000 estados para calcular esa estabilidad global de forma rigurosa (autovalores), aprovechamos que **ya tenéis `solve_pressure` implementado y validado** — que resuelve exactamente ese sistema acoplado completo (el algoritmo de Thomas del Apéndice, Ecs. A1–A11) — y usamos el propio criterio que usa el paper para detectar la inestabilidad:

> *"The solid line phase curve... has a positive slope at $x=0$; this indicates that the traveling-wave propagation is directed outward at the stapes. In other words, the cochlea is emitting sound for this unstable condition."* (paper, Sección III, comentando la Fig. 7)

Es decir: **pendiente de fase positiva en la base ($x\approx0$) = inestable**. Barremos $\gamma$ con `solve_pressure` y buscamos en qué $\gamma$ aparece esa pendiente positiva — el mismo diagnóstico que usa el propio paper, aplicado sistemáticamente en vez de solo a los tres valores puntuales de su Fig. 7.

**Esto no reemplaza** el análisis de autovalores del sistema completo (eso seguiría siendo lo único que *explica* con rigor matemático por qué ocurre a ese valor) — es un contraste más barato, con las herramientas que ya existen, para ver si el fenómeno es reproducible y a qué $\gamma$ aparece.

In [6]:
import numpy as np
import matplotlib.pyplot as plt
from cochleasim.models.cochlear_model import solve_pressure

plt.rcParams['figure.dpi'] = 110

## 1. Carga de parámetros

Igual que en los notebooks anteriores: intenta `CAT_PARAMS` del repositorio, si no usa el respaldo del CSV corregido.

In [4]:
try:
    from cochleasim.models.params_loader import CAT_PARAMS
    params = CAT_PARAMS
    USE_REPO_PARAMS = True
    print("Usando CAT_PARAMS del repositorio.")
except ImportError:
    USE_REPO_PARAMS = False
    print("No se encontró cochleasim.params_loader — usando parámetros de respaldo (CSV corregido).")

if not USE_REPO_PARAMS:

    def _param(A, b_exp, c=0.0):
        return lambda x: A * np.exp(b_exp * np.asarray(x, dtype=float)) + c

    class _FallbackParams:
        k1 = staticmethod(_param(1.10e9, -4))
        c1 = staticmethod(_param(1500, -2, 20))
        m1 = staticmethod(_param(3.00e-3, 0))
        k2 = staticmethod(_param(7.00e6, -4.4))
        c2 = staticmethod(_param(10, -2.2))
        m2 = staticmethod(_param(5.00e-4, 1))
        k3 = staticmethod(_param(1.00e7, -4))
        c3 = staticmethod(_param(2, -0.8))
        k4 = staticmethod(_param(6.15e8, -4))
        c4 = staticmethod(_param(1040, -2))
        gamma = staticmethod(_param(1.0, 0))
        g = staticmethod(_param(1.0, 0))
        b = staticmethod(_param(0.4, 0))
        L = staticmethod(_param(2.5, 0))

    params = _FallbackParams()

Usando CAT_PARAMS del repositorio.


## 2. Adaptador a tu `solve_pressure` real -- **AJUSTAR ANTES DE EJECUTAR EL RESTO**

Esta es la única celda que necesitas tocar. El resto del notebook llama siempre a `run_solve_pressure(freq_hz, gamma_value)` y no le importa cómo esté implementada por dentro, pero necesita:

- `freq_hz`: frecuencia del estímulo en Hz.
- `gamma_value`: valor escalar de $\gamma$ a forzar (constante en $x$, igual que en la Fig. 7 del paper: $\gamma$ es global, no depende de $x$).
- Devuelve un `dict` con, como mínimo:
  - `x`: array de posiciones a lo largo de la cóclea (cm), longitud $N$.
  - `Pd`: array **complejo**, $P_d(x)$ para esa frecuencia y ese $\gamma$.
  - `xi_b`: array **complejo**, $\xi_b(x)$ para esa frecuencia y ese $\gamma$.

Si tu `CochlearParams` no permite forzar $\gamma$ constante directamente, la forma más simple suele ser construir una copia de `params` con `gamma` reemplazado por una función constante, y pasar esa copia a tu `solve_pressure` -- ver el ejemplo comentado dentro de la función.

In [7]:

# Comprobación rápida -- debe ejecutarse sin excepción antes de seguir
_test = solve_pressure(1600.0, 1.0)
assert 'x' in _test and 'Pd' in _test and 'xi_b' in _test, "El dict devuelto debe incluir x, Pd, xi_b"
print("run_solve_pressure OK -- N =", len(_test['x']), "puntos, x en [",
      _test['x'][0], ",", _test['x'][-1], "] cm")

TypeError: solve_pressure() missing 1 required positional argument: 'params'

## 3. Diagnóstico 1 — pendiente de fase en la base (criterio exacto del paper)

Replicamos el criterio textual de la Sección III: para el estímulo de **1.6 kHz** (el mismo que usa la Fig. 7), calculamos la fase de $P_d(x)$ en los primeros puntos ($x\approx0$, la base) y ajustamos una recta. Si la pendiente es negativa, la onda viaja hacia el ápice (normal, estable). Si se vuelve positiva, la onda viaja hacia la base -- el criterio de inestabilidad del paper.

Barremos $\gamma$ finamente entre 0 y 1.3 para localizar el cruce de signo.

In [ ]:
def phase_slope_at_base(freq_hz, gamma_value, n_points=5):
    """Pendiente (rad/cm) de la fase de Pd(x) ajustada en los primeros n_points
    puntos de la base. Positiva -> onda saliente -> diagnóstico de inestabilidad
    (mismo criterio que el paper para la Fig. 7)."""
    res = run_solve_pressure(freq_hz, gamma_value)
    x = np.asarray(res['x'][:n_points], dtype=float)
    phase = np.unwrap(np.angle(res['Pd'][:n_points]))
    slope = np.polyfit(x, phase, 1)[0]
    return slope

freq_test = 1600.0  # Hz -- igual que la Fig. 7 del paper
gammas_finos = np.linspace(0.0, 1.3, 27)
slopes = [phase_slope_at_base(freq_test, g) for g in gammas_finos]

print(f"{'gamma':>8}  {'pendiente fase base (rad/cm)':>28}")
for g, s in zip(gammas_finos, slopes):
    print(f"{g:8.3f}  {s:28.4f}")

# gamma_crit: primer cruce de signo negativo -> positivo
gamma_crit_fase = None
for g0, g1, s0, s1 in zip(gammas_finos, gammas_finos[1:], slopes, slopes[1:]):
    if s0 < 0 <= s1:
        # interpolación lineal del cruce
        gamma_crit_fase = g0 + (0 - s0) * (g1 - g0) / (s1 - s0)
        break

if gamma_crit_fase is not None:
    print(f"\ngamma_crit (cruce de pendiente de fase) ~= {gamma_crit_fase:.3f}")
    print("Referencia del paper: gamma = 1.12")
else:
    print("\nNo se encontró cruce de signo en el rango barrido -- amplía 'gammas_finos'.")

### Gráfica: pendiente de fase en la base vs. $\gamma$

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4.5))
ax.plot(gammas_finos, slopes, marker='o', ms=3)
ax.axhline(0, color='gray', lw=0.8)
if gamma_crit_fase is not None:
    ax.axvline(gamma_crit_fase, color='C1', ls='--', label=f'cruce ~ {gamma_crit_fase:.3f}')
ax.axvline(1.12, color='C3', ls=':', label='gamma=1.12 (paper)')
ax.set_xlabel('gamma')
ax.set_ylabel('pendiente de fase en la base (rad/cm)')
ax.set_title(f'Criterio de inestabilidad del paper (Sec. III) -- f={freq_test:.0f} Hz')
ax.legend()
plt.tight_layout()
plt.savefig('gamma_sweep_fase_base.png', dpi=150)
plt.show()

## 4. Diagnóstico 2 — crecimiento del pico de $|\xi_b(x)|$

Segundo diagnóstico, independiente del primero: cerca de la inestabilidad, el pico de la curva de sintonía debería crecer muy rápidamente (divergiendo) al acercarse a $\gamma_\text{crit}$ -- es la manifestación en magnitud de acercarse a un polo sobre el eje imaginario. Lo miramos para las mismas cuatro frecuencias de las Figs. 4-6 del paper (0.4, 1.6, 6.4, 25.6 kHz).

In [ ]:
frecuencias_paper = [400.0, 1600.0, 6400.0, 25600.0]
gammas_finos2 = np.linspace(0.0, 1.3, 27)

picos = {f: [] for f in frecuencias_paper}
for f in frecuencias_paper:
    for g in gammas_finos2:
        res = run_solve_pressure(f, g)
        picos[f].append(np.max(np.abs(res['xi_b'])))

fig, ax = plt.subplots(figsize=(7, 4.5))
for f in frecuencias_paper:
    ax.semilogy(gammas_finos2, picos[f], marker='o', ms=3, label=f'{f/1000:.1f} kHz')
ax.axvline(1.12, color='C3', ls=':', label='gamma=1.12 (paper)')
ax.set_xlabel('gamma')
ax.set_ylabel('pico |xi_b(x)| (escala log)')
ax.set_title('Crecimiento del pico de sintonía al acercarse a la inestabilidad')
ax.legend(fontsize=8)
plt.tight_layout()
plt.savefig('gamma_sweep_pico_xib.png', dpi=150)
plt.show()

# gamma al que el pico se dispara -- umbral arbitrario: 20x el valor en gamma=1
print(f"{'freq (Hz)':>10}  {'pico en gamma=1':>16}  {'gamma primer disparo (>20x)':>28}")
for f in frecuencias_paper:
    arr = np.array(picos[f])
    idx_g1 = np.argmin(np.abs(gammas_finos2 - 1.0))
    base = arr[idx_g1]
    idx_disparo = np.where(arr > 20*base)[0]
    g_disparo = gammas_finos2[idx_disparo[0]] if len(idx_disparo) else np.nan
    print(f"{f:10.0f}  {base:16.4g}  {g_disparo:28.3f}")

## 5. Reproducción directa de la Fig. 7 del paper

Réplica exacta: magnitud y fase de $\xi_b(x)$ a 1.6 kHz para $\gamma=0$ (pasiva), $\gamma=1$ (normal) y $\gamma=1.12$ (inestable, según el paper) -- para comparar visualmente contra la Fig. 7 original.

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(7, 6), sharex=True)
for g, label in [(0.0, 'gamma=0 (pasiva)'), (1.0, 'gamma=1 (normal)'), (1.12, 'gamma=1.12 (inestable?)')]:
    res = run_solve_pressure(1600.0, g)
    x = np.asarray(res['x'])
    xi_b = np.asarray(res['xi_b'])
    mag_dB = 20*np.log10(np.abs(xi_b) / 1e-9)  # dB re: 1 nm, igual que el paper
    phase_cycles = np.unwrap(np.angle(xi_b)) / (2*np.pi)
    axes[0].plot(x / x[-1], mag_dB, label=label)
    axes[1].plot(x / x[-1], phase_cycles, label=label)

axes[0].set_ylabel('|xi_b| (dB re: 1 nm)')
axes[1].set_ylabel('fase (ciclos)')
axes[1].set_xlabel('distancia desde el estribo (x/L)')
axes[0].set_title('Réplica de la Fig. 7 del paper -- f=1.6 kHz')
axes[0].legend(fontsize=8)
plt.tight_layout()
plt.savefig('gamma_sweep_fig7_replica.png', dpi=150)
plt.show()

print("Comprueba visualmente: en la curva de fase (panel inferior), la pendiente en x/L=0")
print("debería volverse positiva para gamma=1.12 si tu solve_pressure reproduce el mismo")
print("fenomeno de inestabilidad que reporta el paper.")

## 6. Tabla comparativa final

Reúne los $\gamma_\text{crit}$ obtenidos por los dos diagnósticos de este notebook, junto con el rango que salió del análisis local del Bloque 2 y el valor del paper, para dejarlo documentado en la memoria.

In [ ]:
print(f"{'Método':<45} {'gamma_crit':>12}")
print("-"*58)
print(f"{'Paper (Neely & Kim 1986, Fig. 7)':<45} {'1.12':>12}")
print(f"{'Bloque 2 -- local, Zp(x) por posicion':<45} {'2.8 - 3.6':>12}  (rango segun x/L, NO comparable directamente)")
if gamma_crit_fase is not None:
    print(f"{'Este notebook -- cruce pendiente de fase':<45} {gamma_crit_fase:>12.3f}")
else:
    print(f"{'Este notebook -- cruce pendiente de fase':<45} {'sin cruce en rango':>12}")
print(f"{'Este notebook -- disparo de pico |xi_b| (ver tabla sec. 4)':<45}")

## 7. Notas para la interpretación

- Si el $\gamma_\text{crit}$ de este notebook sale **razonablemente cerca de 1.12** (aunque no exacto -- el criterio de "pendiente positiva" es cualitativo, no una definición matemática exacta de polo), es una confirmación fuerte de que `solve_pressure` reproduce correctamente la física de inestabilidad global del paper, con las herramientas que ya teníais construidas.
- Si sale claramente distinto, antes de dudar de `solve_pressure` revisa primero: (a) que `gamma_value` se esté aplicando como constante en **todo** $x$ (el paper es explícito en que $\gamma$ es independiente de $x$ para este experimento, Sección II del paper), y (b) que la resolución espacial ($N$) sea suficiente cerca de la base, donde la sensibilidad a $\gamma$ es mayor.
- Este notebook **no sustituye** el análisis de autovalores del sistema completo -- solo indica *si* y *aproximadamente dónde* aparece el fenómeno, no *por qué* matemáticamente. Si en el futuro decidís abordar el sistema en espacio de estados completo, este resultado sirve como referencia para contrastar ese análisis más riguroso.